In [1]:
""""
Summarize fuel treatment acres within fire perimeters
Maxwell.Cook@colostate.edu
"""

# imports
import os, sys

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from __functions import *  # imports all custom functions

# local data directories
datadir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/data/'
projdir = os.path.dirname(os.getcwd())
print(projdir)

print("Ready !")

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/valuation
Ready !


In [2]:
# load the incidents / perimeters
mtbs = os.path.join(projdir, 'data/spatial/wf_incidents_2014_2022_mtbs_perims.gpkg')
mtbs = gpd.read_file(mtbs)
print(f"Processing for {len(mtbs)} fires.")
print(f"CRS: {mtbs.crs}")
# clean the dataframe just a bit for processing
mtbs = mtbs[['Event_ID','Incid_Name','Ig_Date','BurnBndAc','geometry']]
mtbs.head()

Processing for 3494 fires.
CRS: EPSG:4326


,Event_ID,Incid_Name,Ig_Date,BurnBndAc,geometry
0,NJ3973407472120220619,MULLICA RIVER FIRE,6/19/22 0:00,13082,"MULTIPOLYGON (((-74.69252 39.68403, -74.6923 3..."
1,NC3565607641720220619,FEREBEE ROAD,6/19/22 0:00,1954,"MULTIPOLYGON (((-76.42425 35.65565, -76.42447 ..."
2,FL2576508042220220329,137 AVE,3/29/22 0:00,857,"MULTIPOLYGON (((-80.43589 25.77237, -80.43507 ..."
3,FL2632608099820220328,INTERCEPTOR,3/28/22 0:00,4143,"MULTIPOLYGON (((-81.08343 26.29129, -81.08347 ..."
4,MS3143909088120220304,HWY 556 - BUDE MS,3/4/22 0:00,663,"MULTIPOLYGON (((-90.87669 31.45572, -90.87648 ..."


In [3]:
# load the fuel treatment data
fp = os.path.join(projdir, 'data/spatial/treatments/twig_treatments_1km.gpkg')
trts = gpd.read_file(fp)
# convert date fields and extract the year
trts['treatment_date'] = pd.to_datetime(trts['treatment_date'], unit='ms', errors='coerce')
trts['actual_completion_date'] = pd.to_datetime(trts['actual_completion_date'], unit='ms', errors='coerce')
trts['year_comp'] = trts['actual_completion_date'].dt.year
# check on the structure
print(f"Columns:\n{trts.columns}")
print(f"CRS: {trts.crs}")
trts[['Event_ID', 'unique_id', 'treatment_date', 'actual_completion_date', 'year_comp', 'type', 'acres']].head()

Columns:
Index(['type', 'date_source', 'identifier_database', 'fund_source', 'state',
       'SHAPE__Area', 'unique_id', 'actual_completion_date', 'agency',
       'date_current', 'twig_category', 'treatment_date', 'name', 'acres',
       'category', 'SHAPE__Length', 'objectid', 'index_right', 'Event_ID',
       'Fire_Name', 'activity', 'total_cost', 'cost_per_uom', 'fund_code',
       'uom', 'method', 'equipment', 'activity_code', 'error', 'geometry',
       'year_comp'],
      dtype='object')
CRS: EPSG:3857


,Event_ID,unique_id,treatment_date,actual_completion_date,year_comp,type,acres
0,NC3565607641720220619,126344-6325503,2021-09-16,2021-09-16,2021.0,Mastication,961.308524
1,NC3565607641720220619,126342-6325503,2021-09-16,2021-09-16,2021.0,Mastication,961.308524
2,NC3565607641720220619,126343-6325503,2021-09-16,2021-09-16,2021.0,Mastication,961.308524
3,NC3565607641720220619,137998-6325504,2022-08-01,2022-08-01,2022.0,Mowing,961.308524
4,NC3565607641720220619,138000-6325504,2022-08-01,2022-08-01,2022.0,Mowing,961.308524


In [4]:
trts['type'].unique()

array(['Mastication', 'Mowing', 'Broadcast Burn', 'Machine Pile Burn',
       'Thinning', None, 'Fire Use', 'Machine Pile', 'Biomass Removal',
       'Lop and Scatter', 'Grazing', 'N/A', 'Chemical', 'Hand Pile Burn',
       'Hand Pile', 'Chipping', 'Crushing', 'Mastication/Mowing',
       'Jackpot Burn', 'Seeding', 'Biological', 'Chemical Treatment',
       'Preparation'], dtype=object)

In [5]:
# crop treatments to the fire perimeters
# buffer perimeters 1km
print("Buffering fire perimeters ...")
buffer = mtbs.copy().to_crs(trts.crs) # match CRS to treatments
buffer['geometry'] = buffer.geometry.buffer(1000)
buffer['fire_buffer_acres'] = buffer.geometry.area / 4046.86 # recalculate mtbs acres with buffer
buffer = buffer.to_crs(trts.crs) # now match the treatment CRS
# loop through fires, crop treatment data to bounds
trt_clips = [] # to store the results
for idx, fire_row in tqdm(buffer.iterrows(), total=buffer.shape[0], desc='Processing Fires'):
    fire_id = fire_row['Event_ID']
    fire_acres = fire_row['fire_buffer_acres']
    fire_geom = fire_row['geometry']

    # select overlapping treatments
    trts_fire = trts[trts['Event_ID'] == fire_id]

    if not trts_fire.empty:
        # Clip treatments to the fire perimeter buffer
        clipped = gpd.clip(trts_fire, fire_geom)
        clipped['fire_buffer_acres'] = fire_acres
        trt_clips.append(clipped)
        del clipped, trts_fire

# Combine all clipped results into one GeoDataFrame
trts_fire = gpd.GeoDataFrame(pd.concat(trt_clips, ignore_index=True), crs=trts.crs)
del trt_clips

# Recalculate area
trts_fire['gis_acres'] = trts_fire.geometry.area / 4046.86
trts_fire.head()

Buffering fire perimeters ...


Processing Fires:   0%|          | 0/3494 [00:00<?, ?it/s]

,type,date_source,identifier_database,fund_source,state,SHAPE__Area,unique_id,actual_completion_date,agency,date_current,...,fund_code,uom,method,equipment,activity_code,error,geometry,year_comp,fire_buffer_acres,gis_acres
0,Mastication,act_comp_dt,NFPORS,No Funding Code,NC,3.769421e+06,144542-6325505,2023-08-11,FWS,1.698775e+12,...,None,None,None,None,None,None,"MULTIPOLYGON (((-8508413.872 4259288.306, -850...",2023.0,7850.130151,3.595650
1,Mastication,act_comp_dt,NFPORS,No Funding Code,NC,3.769421e+06,144543-6325505,2023-08-11,FWS,1.698775e+12,...,None,None,None,None,None,None,"MULTIPOLYGON (((-8508413.872 4259288.306, -850...",2023.0,7850.130151,3.595650
2,Mastication,act_comp_dt,NFPORS,No Funding Code,NC,5.914354e+06,126344-6325503,2021-09-16,FWS,1.634591e+12,...,None,None,None,None,None,None,"MULTIPOLYGON (((-8508413.872 4259288.306, -850...",2021.0,7850.130151,4.531692
3,Mastication,act_comp_dt,NFPORS,No Funding Code,NC,5.914354e+06,126342-6325503,2021-09-16,FWS,1.634591e+12,...,None,None,None,None,None,None,"MULTIPOLYGON (((-8508413.872 4259288.306, -850...",2021.0,7850.130151,4.531692
4,Mastication,act_comp_dt,NFPORS,No Funding Code,NC,5.914354e+06,126343-6325503,2021-09-16,FWS,1.634591e+12,...,None,None,None,None,None,None,"MULTIPOLYGON (((-8508413.872 4259288.306, -850...",2021.0,7850.130151,4.531692


In [6]:
# Flatten (dissolve) by Fire ID, Treatment Type, and Year Completed
flat = trts_fire[trts_fire['type'] != 'None'].dissolve(by=['Event_ID','type','year_comp']).reset_index()
flat['type'].unique() # check what treatment types we're working with

array(['Broadcast Burn', 'Biomass Removal', 'Chemical', 'Lop and Scatter',
       'Machine Pile', 'Thinning', 'N/A', 'Fire Use', 'Chipping',
       'Machine Pile Burn', 'Chemical Treatment', 'Grazing', 'Biological',
       'Hand Pile Burn', 'Mastication', 'Mowing', 'Crushing',
       'Mastication/Mowing', 'Jackpot Burn', 'Seeding', 'Hand Pile',
       'Preparation'], dtype=object)

In [8]:
# subset to our treatments of interest
trt_types_keep = [
    'Thinning', 'Machine Pile', 'Hand Pile', 'Mastication',
    'Broadcast Burn', 'Jackpot Burn', 'Fire Use',
    'Machine Pile Burn', 'Hand Pile Burn',
]

# filter our dissolved data
flat_ = flat[flat['type'].isin(trt_types_keep)].copy()
flat_['trt_gis_acres'] = flat_.geometry.area / 4046.86 # recalculate the GIS acres
# keep only a few attributes ...
flat_c = flat_[['Event_ID', 'type', 'year_comp', 'trt_gis_acres', 'fire_buffer_acres', 'geometry']].copy()
print(flat_c['type'].unique())
flat_c.head()

['Broadcast Burn' 'Machine Pile' 'Thinning' 'Fire Use' 'Machine Pile Burn'
 'Hand Pile Burn' 'Mastication' 'Jackpot Burn' 'Hand Pile']


,Event_ID,type,year_comp,trt_gis_acres,fire_buffer_acres,geometry
0,AL3267208704520220215,Broadcast Burn,2008.0,2342.745541,8806.929408,"POLYGON ((-9691170.724 3850413.718, -9691236.3..."
1,AL3267208704520220215,Broadcast Burn,2012.0,2063.940812,8806.929408,"POLYGON ((-9692425.934 3855412.437, -9692385.0..."
2,AL3267208704520220215,Broadcast Burn,2013.0,3340.910059,8806.929408,"POLYGON ((-9687893.187 3853382.023, -9687934.8..."
3,AL3267208704520220215,Broadcast Burn,2022.0,6500.765246,8806.929408,"MULTIPOLYGON (((-9690538.977 3849919.345, -969..."
4,AL3267208704520220215,Broadcast Burn,2024.0,3340.910059,8806.929408,"POLYGON ((-9687893.187 3853382.023, -9687934.8..."


In [14]:
# now create the single dissolved layer by fire ID
# add the treatment years as a list
flat_c_fire = (
    flat_c
    .dissolve(
        by=["Event_ID", "type"], # fire/treatment type
        aggfunc={
            "year_comp": lambda x: sorted(set(x.dropna().astype(int))), # list of treatment years
            "fire_buffer_acres": "first"   # keep the fire acres as a column in the output
        }
    )
    .reset_index()
)

# clip again to fire buffers
flat_c_fire = gpd.clip(flat_c_fire, buffer)
# recalculate the treatment acres
flat_c_fire['treatment_acres'] = flat_c_fire.geometry.area / 4046.86
# calculate the percent treated by treatment type
flat_c_fire['fire_pct_treated'] = (flat_c_fire['treatment_acres'] / flat_c_fire['fire_buffer_acres']) * 100
print(len(flat_c_fire))
flat_c_fire.head()

2989


,Event_ID,type,geometry,year_comp,fire_buffer_acres,treatment_acres,fire_pct_treated
982,FL2523008087020180726,Broadcast Burn,"POLYGON ((-8999828.771 2901862.328, -8999833.5...","[2003, 2005, 2009, 2012, 2021, 2022, 2023]",3964.692412,3823.795434,96.446207
983,FL2563608053120200324,Broadcast Burn,"POLYGON ((-8968431.581 2954251.261, -8968326.6...","[2007, 2010, 2011, 2014, 2015, 2020]",3989.602204,1867.122251,46.799710
985,FL2574708088120200507,Broadcast Burn,"MULTIPOLYGON (((-9013628.387 2959572.987, -901...","[2009, 2010, 2014, 2015, 2020]",49194.064456,4771.631530,9.699608
984,FL2565208050520200419,Broadcast Burn,"POLYGON ((-8961154.132 2962399.844, -8961154.1...",[2021],5940.621994,109.192810,1.838070
992,FL2595108122020170318,Broadcast Burn,"MULTIPOLYGON (((-9042749.612 2983850.007, -904...","[2003, 2004, 2006, 2008, 2009, 2010, 2011, 201...",46986.459872,46449.622228,98.857463


In [20]:
# Join in the unique number of treatments by type
# Count the unique treatments of each type
n_trts = (
    trts_fire
    .groupby(["Event_ID", "type"])
    .agg(n_treatments=("year_comp", "nunique"))
    .reset_index()
)
# join to the dataframe
flat_c_fire_ = pd.merge(flat_c_fire, n_trts, on=["Event_ID", "type"], how="left")
# tidy up some of the columns
flat_c_fire_ = flat_c_fire_[[
    'Event_ID', 'type', 'treatment_acres',
    'fire_buffer_acres', 'fire_pct_treated',
    'year_comp', 'n_treatments', 'geometry'
]]
flat_c_fire_.rename(columns={
    'type': 'treatment_type',
    'year_comp': 'treatments_years'
}, inplace=True)
flat_c_fire_.head()

,Event_ID,treatment_type,treatment_acres,fire_buffer_acres,fire_pct_treated,treatments_years,n_treatments,geometry
0,FL2523008087020180726,Broadcast Burn,3823.795434,3964.692412,96.446207,"[2003, 2005, 2009, 2012, 2021, 2022, 2023]",7,"POLYGON ((-8999828.771 2901862.328, -8999833.5..."
1,FL2563608053120200324,Broadcast Burn,1867.122251,3989.602204,46.799710,"[2007, 2010, 2011, 2014, 2015, 2020]",6,"POLYGON ((-8968431.581 2954251.261, -8968326.6..."
2,FL2574708088120200507,Broadcast Burn,4771.631530,49194.064456,9.699608,"[2009, 2010, 2014, 2015, 2020]",5,"MULTIPOLYGON (((-9013628.387 2959572.987, -901..."
3,FL2565208050520200419,Broadcast Burn,109.192810,5940.621994,1.838070,[2021],1,"POLYGON ((-8961154.132 2962399.844, -8961154.1..."
4,FL2595108122020170318,Broadcast Burn,46449.622228,46986.459872,98.857463,"[2003, 2004, 2006, 2008, 2009, 2010, 2011, 201...",15,"MULTIPOLYGON (((-9042749.612 2983850.007, -904..."


In [21]:
flat_c_fire_['fire_pct_treated'].describe()

count    2989.000000
mean       13.879757
std        22.915907
min         0.000039
25%         0.395470
50%         2.333407
75%        14.071321
max       100.000000
Name: fire_pct_treated, dtype: float64

In [26]:
# save this long format table
out_fp = os.path.join(projdir, 'data/tabular/MTBS_TWIG_summary_long-format.csv')
flat_c_fire_.to_csv(out_fp, index=False)

In [25]:
# make the table wide to easily join with model data
flat_c_fire_w = (
    flat_c_fire_
    .pivot_table(
        index="Event_ID",
        columns="treatment_type",
        values="fire_pct_treated",   # proportion of fire treated
        aggfunc="sum",               # if multiple rows per fire/type, sum them
        fill_value=0
    )
    .reset_index()
)
# merge back the other attributes
print(len(flat_c_fire_))
print(len(flat_c_fire_w))
flat_c_fire_w.head()

2989
1227


treatment_type,Event_ID,Broadcast Burn,Fire Use,Hand Pile,Hand Pile Burn,Jackpot Burn,Machine Pile,Machine Pile Burn,Mastication,Thinning
0,AL3267208704520220215,74.122273,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000
1,AL3274708707120221010,13.535626,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000
2,AL3277908693520190402,83.982393,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000
3,AL3317108611720161113,10.885592,0.0,0.0,0.0,0.0,0.192194,0.0,0.0,2.123935
4,AL3326608609720161012,65.271042,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,10.370532


In [28]:
# save the wide format table (just treatment percentages)
out_fp = os.path.join(projdir, 'data/tabular/MTBS_TWIG_summary_wide-format.csv')
flat_c_fire_w.to_csv(out_fp, index=False)
print(f"Saved to: {out_fp}")

Saved to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/valuation/data/tabular/MTBS_TWIG_summary_wide-format.csv


In [23]:
# check on the cameron peak fire as an example
flat_c_fire_w[flat_c_fire_w['Event_ID']=='CO4060910587920200813']

treatment_type,Event_ID,Broadcast Burn,Fire Use,Hand Pile,Hand Pile Burn,Jackpot Burn,Machine Pile,Machine Pile Burn,Mastication,Thinning
355,CO4060910587920200813,1.048261,0.0,0.0,0.0,0.0,1.599927,1.586057,0.0,1.313764
